# Stores

**Prerequisites:** `01_what_memory_is.ipynb`. From `03_tools`: `02_the_registry.ipynb` is useful but not required.

Notebook 1 ended with a file holding everything the agent had learned, and a problem: a file cannot hand back *part* of itself. You either load every fact ever recorded into the context window, or you load none of them.

What is needed is a way to ask for the few facts that bear on the question in front of you. That is a query — so this notebook is about deciding **where a memory belongs**, such that the query you will need later is one the store can actually serve.

One thing decides it, and it is settled when the memory is *written*: **what will be asked of it.** The shape the memory ends up in follows from that; it is not a criterion of its own.

| Store | What it will be asked | Example |
|---|---|---|
| `KeyValue` | looked up by name | "what is this user's timezone" |
| `Records` | filtered, ordered, counted | "which failed most recently", "how many last week" |
| `Journal` | searched by literal, and by resemblance | "where did `PGRST204` come from", "has this happened before" |
| `Graph` | walked | "who else touched this, and what did they decide" |

`Records` and `Journal` describe the same world, and what separates them is that one gets counted and the other gets searched. Who wrote them is **not** the criterion — but it does explain why each arrives in the shape it does. A deploy pipeline emits rows on a schema designed before the first deploy ran, so filtering is cheap. The agent's notes on *why* something broke could never have had a schema designed in advance, so what is available is prose.

**One thing to keep in mind throughout:** none of these is an upgrade on the one before it. A dict answers its question exactly, instantly and correctly; a vector index answers that same question *approximately*, which is strictly worse. Later in the list does not mean better.

**By the end you'll have:**
- Chosen between two stores by **what each would be asked** rather than by which one had fields, with neither stripped down to make the other look necessary
- Answered a key lookup with a dict, and seen why most long-term memory should stop there
- Used SQL for filtering, ordering and counting, and found the one query no vector database has at any size
- Learned what FTS5 actually is — a boolean `MATCH` selecting candidates, then **BM25** ranking them — and read the negative scores that prove it
- Measured term frequency, saturation and length normalisation in three real numbers
- Asked a question naming **no field to filter on**, and watched every one of its words return zero from full-text search
- Confirmed that filtering does not rescue it, because narrowing is not answering
- Fused BM25 and cosine by reciprocal rank fusion, watched fusion make an answer **worse**, and traced it to the corpus rather than the method
- Filtered by metadata before ranking, and hit the wrinkle that forces dates to be stored twice
- Walked a graph to answer a question whose answer is in no single entry
- Expired an edge instead of deleting it, so "who owns this now" and "who owned it in July" both still answer
- **Composed** the four, because plenty of questions need two of them — each store taking a turn that either answers or narrows what the next one may look at

## Setup

`stores.py` sits one folder up. Two of these run on real databases:

| What | Needs |
|---|---|
| `KeyValue`, `Records` | nothing (JSON file, SQLite) |
| `Journal` — literal index | nothing (SQLite FTS5) |
| `Journal` — vectors and metadata | `pip install chromadb` |
| embeddings | `GEMINI_API_KEY` — Groq serves no embedding model |
| `Graph` | `pip install neo4j` **and a running server**: `NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD` |

Neo4j has no embedded mode, so **this is the only notebook in the repo that cannot run on its own** — it needs a service up somewhere. A free instance at [console.neo4j.io](https://console.neo4j.io) takes a couple of minutes; see `.env.example`.

That requirement is part of the lesson. A graph database is infrastructure, and the cost of running one belongs in the decision to use one.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))

import json
import shutil

from stores import Graph, Journal, KeyValue, Records

TODAY = "2026-08-22"

# This notebook owns `store/`, and both `Records.write` and `Journal.write` append. Without
# clearing first, a second run of the notebook prints twelve rows where the outputs below say
# six. Only the three things seeded here are removed; `store/memories`, which `03_operations`
# builds, is left alone.
STORE = Path("store")
STORE.mkdir(exist_ok=True)
(STORE / "profile.json").unlink(missing_ok=True)
(STORE / "records.db").unlink(missing_ok=True)
shutil.rmtree(STORE / "journal", ignore_errors=True)

## Question 1 — "what is this user's timezone?"

A name, `timezone`, with one value under it. The agent was told it once and will look it up by that name forever.

That is a dict. Nothing else needed.

In [2]:
profile = KeyValue(STORE / "profile.json")

profile.write("timezone", "Asia/Kolkata")
profile.write("examples", "Python")

print("timezone :", profile.read("timezone"))
print("missing  :", repr(profile.read("shell")))
print("reopened :", KeyValue(STORE / "profile.json").read("timezone"))

timezone : Asia/Kolkata
missing  : None
reopened : Asia/Kolkata


`read("shell")` returns `None` — not a guess, not the nearest thing. Hold on to that; by question 4 it will look like a luxury.

**Most of what an agent needs to remember stops here.** A timezone, a preferred language, a project root: each is a fact with a name. Reaching past a dict for any of them buys nothing and costs exactness — the vector index later would answer `timezone` with whatever most *resembles* a timezone, which is a worse answer to a question that had a perfect one.

**Where it should actually live.** In production this goes in the same durable store as everything else structured — a `profile` table in SQLite here, Postgres at scale — which gives you atomic single-key writes, many readers, and one thing to back up instead of a scattering of JSON files. It is still a key and a value; only the guarantees change.

Worth pairing with the note in notebook 1, because the two pull in opposite directions. A *conversation* belongs in Redis: hot, short-lived, shared between workers, expiring on its own. A *profile* belongs in a durable store: cold, permanent, and the thing you would be upset to lose. Both are keys and values, and the interface tells you nothing about which to use — the lifetime does.

## Two stores for the same world

Everything else in this notebook concerns one team's services. It lands in two different stores, and what separates them is **what each will be asked**.

**`Records`** will be filtered, ordered and counted. That is possible because every row has a time, a kind, a status and a service — a schema somebody designed before the first deploy ever ran, which is what a pipeline emitting rows makes possible.

**`Journal`** will be searched, by literal and by resemblance. There are no columns to design here: every conclusion is about something different, and a schema general enough to hold them all would be too generic to retrieve on. The prose is the thing you search.

In [3]:
records = Records(STORE / "records.db")

records.write_many([
    ("2026-07-02", "incident", "resolved", "checkout-api", "Connection pool exhausted during peak traffic."),
    ("2026-08-11", "deploy",   "ok",       "checkout-api", "Deployed 4.2.0."),
    ("2026-08-12", "deploy",   "failed",   "checkout-api", "Rolled back after twelve minutes."),
    ("2026-08-14", "deploy",   "ok",       "billing-api",  "Deployed 2.9.1."),
    ("2026-08-18", "query",    "failed",   "billing-api",  "Schema cache error."),
    ("2026-08-19", "deploy",   "failed",   "billing-api",  "Rolled back. Error rate hit 4 percent."),
])

journal = Journal(STORE / "journal")

journal.write([
    ("2026-07-02", "checkout-api", "tool_output",
     "The pool ran dry under peak traffic; raising max connections to 60 cleared it."),
    ("2026-08-11", "checkout-api", "user_said",
     "Releases go out on Tuesdays unless someone objects."),
    ("2026-08-12", "checkout-api", "tool_output",
     "The migration held a lock on the users table for about forty seconds while it rebuilt an index."),
    ("2026-08-14", "billing-api", "concluded",
     "Billing runs its own migrations on a separate schedule."),
    ("2026-08-18", "billing-api", "tool_output",
     "PostgREST returned PGRST204: column 'invoice_ref' not found in schema cache."),
    ("2026-08-19", "billing-api", "user_said",
     "Priya should be paged when the error rate goes above two percent."),
])

print("records:", len(records), "| journal entries:", len(journal))

records: 6 | journal entries: 6


## Question 2 — "which deployment failed most recently, and how many have there been?"

No key to look up. This is a *filter* (`kind`, `status`), an *ordering* (latest first), a *limit* — and then a count.

In [4]:
row = records.recent(kind="deploy", status="failed", limit=1)[0]
print("latest failed deploy:", row["at"], "|", row["service"], "|", row["text"])

print()
print("failed deploys           :", records.count(kind="deploy", status="failed"))
print("everything on billing-api:", records.count(service="billing-api"))

latest failed deploy: 2026-08-19 | billing-api | Rolled back. Error rate hit 4 percent.

failed deploys           : 2
everything on billing-api: 3


The filtering, ordering and counting happened in the database; only the answer crossed into Python.

That `count` is the one query in this notebook with **no equivalent anywhere else**. Chroma has no `COUNT` and no `GROUP BY`, at any corpus size. If your questions include "how many", SQL is not optional — and that is true whether you have six rows or six million.

The journal has fields too, and they answer a different question: not what happened, but **how the agent came to know it**.

In [5]:
for source in ["user_said", "tool_output", "concluded"]:
    print(f"  {source:<12} {journal.count(source=source)}")

print()
print("most recent entry about billing-api:")
print("  ", journal.recent(about="billing-api", limit=1)[0]["text"][:66])

  user_said    2
  tool_output  3
  concluded    1

most recent entry about billing-api:
   Priya should be paged when the error rate goes above two percent.


Two of these came from the user, three from tool output, one the agent concluded on its own. That distinction is worth keeping: one is hearsay, one is observed, and one is the agent's own inference — which is the first thing you want to know when a memory turns out to be wrong.

## Question 3 — "where did that `PGRST204` error come from?"

A literal. Nothing but `PGRST204` will do, and something that merely *resembles* it is worse than useless.

This goes to the journal, because that is where the agent wrote down what it saw. Same rows, different index: FTS5.

In [6]:
for row in journal.find_exact("PGRST204"):
    print(row["at"], "|", row["about"], "|", row["source"])
    print(row["text"])

2026-08-18 | billing-api | tool_output
PostgREST returned PGRST204: column 'invoice_ref' not found in schema cache.


### What FTS5 is actually doing

Calling that "exact matching" is half the story, and the half that is missing matters.

**BM25 is a ranking algorithm. FTS5 is the engine that implements it.** A search runs in two stages:

1. **`MATCH` is boolean**, over an inverted index. A row either contains the token or it does not; the ones that do not are gone. *This* stage is 1/0, and it is why a search for something absent can return **nothing** — an answer no vector index can give.
2. **`bm25()` ranks the survivors** on a continuous scale. SQLite exposes it as the hidden `rank` column, and the values are **negative**, so `ORDER BY rank` ascending puts the best match first.

Here are real scores from a four-row table searched for `PGRST204`:

```
-1.8690e-06    3 occurrences,   7 words
-1.5725e-06    1 occurrence,    3 words
-5.0708e-07    1 occurrence,   92 words
```

Four things go into that score, and three of them are visible in those numbers:

- **Term frequency** — three occurrences beat one.
- **Saturation** — but 3× the occurrences buys only about 1.19× the score, not 3×. Repetition is dampened deliberately.
- **Length normalisation** — the *same single occurrence* scores roughly 3× worse in a 92-word row than in a 3-word row.
- **Rarity** — the one you cannot see above, because all three rows contain the same term and it therefore contributes equally to each. Across a whole corpus it is decisive: a term appearing in nearly every row separates nothing.

That last one is what makes this a *ranking* function rather than a filter.

### Why not let Chroma do the text search?

Chroma offers `where_document={"$contains": ...}`. It is substring matching over raw text, with no ranking at all, and it is wrong in both directions on the cases that matter:

| query | FTS5 | Chroma `$contains` |
|---|---|---|
| `PGRST204` | 1 row | 1 row |
| `PGRST20` | **0** — not a token, so not a match | **1** — matches as a prefix |
| `pgrst204` | **1** — case-insensitive | **0** — misses it |

FTS5 tokenises; `$contains` looks for characters. For a question whose whole point is that `PGRST205` means something other than `PGRST204`, that decides it.

This is also why the journal — not `Records` — is where literals get searched. An agent's note about a failure is largely *the failure string*: `PGRST204`, `max_connections`, `src/billing/reconcile.py`. Those are what a person searches for later, and what dense vectors are worst at, because an embedding collapses surface form into meaning and an identifier has no meaning to collapse to.

## Question 4 — "has this happened before?"

A different shape of question entirely. Something is going wrong now, and all you have is a description:

> *checkout is timing out and the database seems to be blocking*

There is no key. There is no field to filter on either — the sentence does not say it was a deploy, or that it failed, or which date to look near. It is a **symptom**, not a category, and mapping it onto fields is the thing you would need the answer in order to do.

Full-text search cannot help, and not for lack of text. Every word of the question is absent from every entry:

In [7]:
question = "checkout is timing out and the database seems to be blocking"

for term in ["timing", "timeout", "blocking", "checkout", "database"]:
    print(f"  {term:>9}: {len(journal.find_exact(term))} hits")

     timing: 0 hits
    timeout: 0 hits
   blocking: 0 hits
   checkout: 0 hits
   database: 0 hits


Zero across the board — including `checkout`, which is in the `about` field of three entries but in nobody's prose.

Filtering does not rescue it either. Suppose you guess it concerns `checkout-api`:

In [8]:
for row in journal.recent(about="checkout-api", limit=5):
    print(f"  {row['at']}  {row['source']:<12} {row['text'][:52]}")

  2026-08-12  tool_output  The migration held a lock on the users table for abo
  2026-08-11  user_said    Releases go out on Tuesdays unless someone objects.
  2026-07-02  tool_output  The pool ran dry under peak traffic; raising max con


Three entries, and the filter has no opinion about which of them explains a timeout. **Narrowing is not answering.**

This is the question that genuinely needs embeddings. An embedding places text so that things which *mean* similar things land near each other, whether or not they share vocabulary — and then "near" is arithmetic.

In [9]:
for score, entry_id, text in journal.find_like(question, k=3):
    print(f"{score:.4f}  {text[:66]}")

0.6670  The migration held a lock on the users table for about forty secon
0.6496  The pool ran dry under peak traffic; raising max connections to 60
0.6263  Billing runs its own migrations on a separate schedule.


Both prior contention incidents, in the right order, sharing not one word with the question.

Now the catch — clearer with every score shown.

In [10]:
scores = journal.find_like(question, k=len(journal))

for score, _id, text in scores:
    print(f"{score:.4f}  {text[:66]}")

print()
print(f"best {scores[0][0]:.4f} | worst {scores[-1][0]:.4f} | "
      f"range {scores[0][0] - scores[-1][0]:.4f} wide")

0.6670  The migration held a lock on the users table for about forty secon
0.6496  The pool ran dry under peak traffic; raising max connections to 60
0.6263  Billing runs its own migrations on a separate schedule.
0.5934  PostgREST returned PGRST204: column 'invoice_ref' not found in sch
0.5807  Releases go out on Tuesdays unless someone objects.
0.5375  Priya should be paged when the error rate goes above two percent.

best 0.6670 | worst 0.5375 | range 0.1295 wide


Two things matter more than the ranking.

**Nothing scores near zero.** The note about paging Priya has no relationship to a timeout, and it still lands in the same neighbourhood as the correct answer. Cosine similarity on real embeddings never uses its theoretical range.

**There is no such thing as "no match".** `k=3` returns three entries whether or not any are relevant — exactly what `read("shell")` refused to do in question 1. A dict can say *I don't have that*. A vector index has no way to say it.

Both come due in notebook 3.

## Question 3, again — could embeddings have handled `PGRST204`?

That entry is in the vector index too. Ask it directly:

In [11]:
for score, _id, text in journal.find_like("PGRST204", k=2):
    print(f"{score:.4f}  {text[:66]}")

0.6852  PostgREST returned PGRST204: column 'invoice_ref' not found in sch
0.5927  Priya should be paged when the error rate goes above two percent.


It found it. So is FTS5 redundant?

No — and what separates them is not a query the index *has*. It is one it does not. `PGRST205` is a different error, and it never happened:

In [12]:
print("FTS5 for PGRST205   :", len(journal.find_exact("PGRST205")), "hits")
print()
print("vectors for PGRST205:")
for score, _id, text in journal.find_like("PGRST205", k=2):
    print(f"  {score:.4f}  {text[:60]}")

FTS5 for PGRST205   : 0 hits

vectors for PGRST205:


  0.6520  PostgREST returned PGRST204: column 'invoice_ref' not found 
  0.5868  Priya should be paged when the error rate goes above two per


FTS5 returns nothing, correctly. The vector index returns a *different error code*, confidently, and scores it right among the scores it gave correct answers on the previous question. No cutoff separates them, because scores rank resemblance and resemblance from two different questions is not on a common scale.

Embeddings capture meaning; an identifier has none. `PGRST204` and `PGRST205` differ by one character and mean unrelated things — exactly the distinction embeddings are built to smooth over.

So for a query you can *tell* is a label, ask the exact index first and fall back only if it finds nothing:

In [13]:
def lookup(term, k=2):
    exact = journal.find_exact(term)
    if exact:
        return [("exact", row["text"]) for row in exact]
    return [(f"{s:.4f}", t) for s, _i, t in journal.find_like(term, k=k)]

for term in ["PGRST204", "PGRST205"]:
    print(term + ":")
    for label, text in lookup(term):
        print(f"  {label:>7}  {text[:60]}")
    print()

PGRST204:
    exact  PostgREST returned PGRST204: column 'invoice_ref' not found 

PGRST205:
   0.6520  PostgREST returned PGRST204: column 'invoice_ref' not found 
   0.5868  Priya should be paged when the error rate goes above two per



The label matters as much as the result: whatever consumes this has to be able to tell *found* from *nearest*.

**But that rule works only because `PGRST204` is a label** and you can see that by looking at it. An ordinary question has no such tell, and this rule would be actively wrong for one — any single word matching would suppress the vector ranking entirely.

## Hybrid search: fusing the two rankings

The general form combines the two *rankings* rather than choosing between them. That is what hybrid search means — BM25 and cosine, fused.

The fusion is done on **ranks**, not scores, and the reason is now concrete: FTS5's BM25 scores are negative and corpus-dependent (`-1.87e-06`), while cosine sits in a narrow positive band (`0.54`–`0.67`). Nothing sensible normalises those into each other, and trying runs into the trap notebook 3 covers, where min-max promotes the least-bad of a bad batch to 1.0. Ranks have no such failure mode:

```
score(d) = sum over each index of  1 / (60 + rank of d in that index)
```

`Journal.find_hybrid` does that and returns both input rankings alongside the fused one, so you can see what each contributed:

In [14]:
for q in [question, "the migration that broke invoice_ref"]:
    fused, lexical, vector = journal.find_hybrid(q, k=3)
    print(q)
    print("   BM25   :", lexical)
    print("   vector :", vector[:4])
    print("   fused  :", [eid for _s, eid, _t in fused])
    print("   ->", fused[0][2][:60])
    print()

checkout is timing out and the database seems to be blocking
   BM25   : [2]
   vector : [3, 1, 4, 5]
   fused  : [2, 3, 1]
   -> Releases go out on Tuesdays unless someone objects.



the migration that broke invoice_ref
   BM25   : [5, 3]
   vector : [5, 4, 3, 2]
   fused  : [5, 3, 4]
   -> PostgREST returned PGRST204: column 'invoice_ref' not found 



**Look at the first one. Fusion made the answer worse.**

The vector index ranked entry 3 first — the migration holding a lock, which is the right answer. BM25 returned entry 2, *"Releases go out on Tuesdays unless someone objects."* Fusion promoted that to the top.

Entry 2 matched on the word **`out`**, from "timing *out*" in the question meeting "go *out* on Tuesdays" in the entry. `find_hybrid` strips closed-class words before querying FTS5, and `out` is not in its stopword list.

Before drawing a lesson from that, it is worth asking why BM25 fell for `out` at all — because it has a defence against exactly this, and the defence is not working here.

**Rarity is the fourth term in the score, and six entries are not enough to estimate it.** `out` appears in one entry of six, so BM25 sees a term in 16% of the corpus and treats it as informative. Give it a real corpus and the same term is crushed:

```
corpus size   entries containing "out"   bm25 of the match
        6              1  (16%)           -1.299
       50             15  (30%)           -0.862
      200            120  (60%)           -1.081e-06
```

At 60% document frequency the term contributes essentially nothing. BM25 is *designed* to discount words that fail to separate documents; it simply cannot tell which those are until it has documents to count.

So the honest reading of the failure above is narrower than it first appears:

- **This particular failure is a small-corpus artifact.** The stopword list is standing in for a job that rarity would do on its own at scale. On six entries you are hand-maintaining what IDF gives you for free on sixty thousand.
- **What does generalise: fusion inherits both inputs.** A lexical index confidently wrong about one entry drags it up the fused ranking however good the vector side was — that is true at any size, and it is why the two rankings are worth inspecting separately rather than trusting the fused one.
- **What also generalises: BM25 expects search terms, not sentences.** Hand it a whole question and every word in it becomes a query term. Asked *"what happened with the rollback"* over these six entries, here is what each word finds on its own:

  ```
  what      ->  nothing          the       ->  entries 1, 3 and 6
  happened  ->  nothing          rollback  ->  nothing
  with      ->  nothing
  ```

  Every word that carries any meaning matches nothing — `rollback` included, because no entry happens to use that word. The only term that hits anything is `the`. So three entries come back on the strength of `the` alone, all scoring `-0.0000`, and which of them lands on top is arbitrary. The ranking looks like a result and is noise.

  Where the stripping should happen has two answers, and they are not interchangeable. Elasticsearch's English analyzer carries a `stop` filter, so words like `the` never reach the index at all. FTS5 carries none — it indexes `the`, `on` and `a` like any other token — so it falls to the query builder, which is why `find_hybrid` holds a stopword list. Keeping them is a defensible default rather than an oversight: IDF already discounts them on a corpus large enough to measure rarity, and stripping breaks exact phrases like *"to be or not to be"*.

None of this argues against hybrid search. It argues that a demonstration this small shows you the machinery without showing you the behaviour — and that the two should not be confused.

### Chroma can do this natively — just not here

Chroma has hybrid search built in: a BM25 sparse index, and an `Rrf` operator that fuses rankings exactly as above.

```python
Search().rank(Rrf([Knn(query=dense_vector),
                   Knn(query="lock users table", key="sparse")]))
```

Asking for it in this notebook returns:

```
NotImplementedError: Search is not implemented for Local Chroma
```

It works on distributed and hosted Chroma, not on the embedded `PersistentClient` running here. Which is a very ordinary reason to end up running two engines, and a more common one than any argument about scale: **the product has the feature; the deployment you are on does not.**

The rule worth carrying: when you pick a database, check what your *deployment* supports, not what the documentation describes.

## Narrowing by metadata

One more thing the vector index does: let a question carry a condition. *"Reads like this"* becomes *"reads like this, and is about checkout-api, and is from August"* — one query with two kinds of constraint in it.

Note that this says nothing about how the engine goes about it. `where` is part of the **question**; whether the condition is resolved before the search, during it, or after is an implementation choice with real consequences, and a section below. This one is about deciding what the condition should say.

In [15]:
recent_checkout = {"$and": [{"about": "checkout-api"}, {"at": {"$gte": 20260801}}]}

for label, where in [("all entries", None), ("checkout-api, since 1 Aug", recent_checkout)]:
    print(label + ":")
    for score, entry_id, text in journal.find_like(question, k=3, where=where):
        print(f"  {score:.4f}  {text[:56]}")
    print()

all entries:
  0.6670  The migration held a lock on the users table for about f
  0.6496  The pool ran dry under peak traffic; raising max connect
  0.6263  Billing runs its own migrations on a separate schedule.

checkout-api, since 1 Aug:
  0.6670  The migration held a lock on the users table for about f
  0.5807  Releases go out on Tuesdays unless someone objects.



That is the query neither of the others expresses alone. SQL filters exactly but has no notion of "reads like this"; a bare vector index ranks by meaning but cannot restrict itself to one service and a date range.

One wrinkle: `{"at": {"$gte": "2026-08-01"}}` raises `ValueError`. Chroma's range operators take numbers only, which is why `at` goes into the metadata as `20260801` while SQL keeps `"2026-08-01"`. Small difference, and it decides how you store every date.

### Where the filter runs

One line of code, three possible implementations, depending on *when* the constraint is applied relative to the search. It makes no difference at six rows. At six million it decides your latency, and whether you get an answer at all.

To see why: a vector index is not a list you scan. It is a graph — **HNSW** — where each vector links to its near neighbours, and a search walks from one to the next, getting closer. It deliberately never looks at most of the data; that is where the speed comes from. A filter interferes with that walk, and the three strategies interfere differently.

#### Selectivity, first

Everything below is stated in terms of **filter selectivity**: the fraction of the collection that passes the filter.

The word runs backwards from the intuition, so pin it down now. A **highly selective** filter is *restrictive* — it lets almost nothing through. `user_id == 4471` might pass 0.01% of rows. A filter of **low selectivity** is loose: `status == "active"` might pass 95%. High selectivity, few survivors. Read it the other way and every rule below inverts.

And the part that actually bites: **selectivity belongs to the query, not to the field.** The same key behaves like a different filter depending on the value it is given.

| Filter | Passes | |
|---|---|---|
| `category == "Electronics"` | ~40% of the catalogue | loose |
| `category == "Left-Handed Ergonomic Keyboards"` | ~0.05% | extremely restrictive |
| `created_at > "2020-01-01"` | nearly everything | loose |
| `created_at > "2026-09-01"` | the last few days | extremely restrictive |

Same column, same operator, four different problems. Which is why a strategy hardcoded once, in your application, is wrong about half the time.

#### Prefiltering — resolve the metadata first, search only what qualifies

A metadata index (inverted index, bitmap, B-tree, SQL predicate) produces the eligible set, and the vector search runs only inside it.

Right by construction: it asks *what are the nearest neighbours among the rows that qualify*, which is the question you meant. Ask for `k` and you get `k` qualifying rows, or every one there is.

Its cost depends on what the eligible rows look like **inside the graph**, not merely how many there are. Restrictive enough and the engine can abandon the graph entirely and compare every eligible vector exactly — fast, and no longer approximate. Restrictive but *scattered*, and the walk has to cross ineligible nodes just to get from one eligible one to the next; a highly selective filter can leave the graph so sparsely populated that traversal degenerates into the brute-force scan it exists to avoid. At the other end, a loose filter buys nothing: you pay for the metadata lookup and still search almost everything.

#### Postfiltering — search everything, then discard the rows that fail

The ANN index returns the global top candidates; the filter removes the ones that do not match. It is the only one of the three that needs no database support at all — pull the top 20 from any vector index and drop the misses with an `if` statement.

The walk is untouched, so vector-search latency is predictable and the index's accuracy is unharmed. The cost lands on recall instead, because the filter is applied to a candidate pool that approximate search has **already truncated**. A qualifying row slightly farther away lost its place to a nearer row that is about to be thrown out. Ask for 10 and get 4 — and the six missing ones were not near-misses, they were never candidates. Search a million articles with a filter for one customer and you can get back nothing at all, because the global top 20 all belong to other customers. It is also plain waste: if the filter removes 95% of the collection, 95% of your distance computations were spent on rows you discard.

The patch is **over-retrieval**: ask the index for far more than you need, filter, keep the first `k` still standing. Size it from the filter's match rate — if roughly 1 row in 10 passes, ask for 100 to end up with about 10; 1 in 100, ask for 1,000. It is an estimate, and it fails in two directions. It does not buy correctness: if the eligible rows sit far below the global top, a bigger pool still never reaches them. And it is not free — every extra candidate is another distance computation, more memory traffic, more network, more reranking. Under a genuinely selective filter the choice collapses to a small pool that is wrong or a large one that is expensive.

#### In-graph filtering — check eligibility while walking

The predicate is evaluated *during* traversal: at each hop the engine weighs vector distance and the metadata condition together, so the search keeps going until it has enough qualifying results rather than being corrected after the fact. Ineligible nodes can still be stepped *through* as connective tissue; they simply never enter the results.

That is why its recall beats postfiltering's. Its cost turns on whether the filter agrees with the vectors. "Written in German" over a corpus where German documents already cluster together is cheap — eligible neighbours sit next to each other. A filter that cuts across meaning is not: the walk keeps landing in regions that are semantically right and ineligible, and has to widen. Push selectivity far enough and it degenerates toward scanning everything, at which point good implementations fall back to exact search over the eligible slice.

The catch is that you cannot write this one yourself. It lives inside the index — Qdrant, Weaviate, Milvus, modern Neo4j. On an engine without it, your choice is the other two.

#### Nobody picks one

Since selectivity changes per query, a fixed strategy is a fixed mistake: always prefilter and `category == "Electronics"` crawls; always postfilter and the left-handed keyboards return nothing, because they never reach the global top. Serious engines therefore run a **cost-based query planner** — before searching, estimate from index statistics how many rows *this* filter value will match, then switch.

In-graph filtering is the modern default in dedicated vector databases, but it has not retired the other two, because some situations answer the question before any planner does:

- **Multi-tenancy.** `org_id == "xyz"` isolates fifty documents. A metadata lookup followed by an exact scan of fifty vectors is instant and needs no graph infrastructure at all. Prefiltering, and nothing else is worth considering.
- **Relational engines.** With `pgvector` the data is already in Postgres, and the SQL planner will use its ordinary indexes to cut the rows down before any distance is computed. Prefiltering by architecture.
- **A near-universal constraint.** `language == "English"` in an English-language corpus removes almost nothing. Postfiltering costs nothing and leaves the index untouched.
- **Correctness before latency.** A permission or tenant filter is not a preference: returning another tenant's row is a bug, and quietly returning three rows when the caller asked for ten can be one too. That argues for prefiltering whatever the latency table says.
- **Prototyping.** Twenty results and an `if` statement. You will not out-engineer a query planner on day one, and at your current row count it does not matter.

Chroma prefilters, and does not offer you the choice. At six entries everything is compared exactly anyway, so none of this shows in the cells above — which is the reason to know it now: the failure mode arrives before you think to go looking for it.

### Where do those filter values come from?

`recent_checkout` above was written by hand. A live agent is handed one string —

> *"what did the tools tell us about billing since the start of August?"*

— and has to end up with a `where` plus the leftover prose to rank on. Three different things produce those:

1. **Scope comes from the caller, never from the question.** Which user, which tenant, which project. If the model can influence those, a prompt can ask for someone else's memories — so they are passed in by the code that owns the request.
2. **Bounds stated in the question come from the model**, as a structured extraction.
3. **Whatever is left stays prose** and gets ranked. "Timing out", "blocking": no field holds those.

The machinery for the second is already built. `03_tools` put the permitted values into the schema with `Literal` so the model reads them before it writes a token, and validated the rules a type cannot express — `field_validator` for one field, `model_validator` for a rule about a pair. All of it applies here one layer up: instead of validating a tool call, it validates a question split into a filter and a search string.

In [16]:
from typing import Literal, Optional

from pydantic import BaseModel, Field, model_validator

from llm import extract


class JournalFilter(BaseModel):
    """A question, split into the part you can filter on and the part you must rank."""

    query: str = Field(description="what to search for, in the asker's own words")
    source: Optional[Literal["user_said", "tool_output", "concluded"]] = Field(
        default=None, description="only if the question says how the agent came to know it")
    since: Optional[int] = Field(default=None, description="YYYYMMDD, only if the question bounds it")
    until: Optional[int] = Field(default=None, description="YYYYMMDD, only if the question bounds it")

    @model_validator(mode="after")
    def range_must_not_run_backwards(self):
        # raise, don't repair -- `extract` hands the message back and the model retries
        if self.since and self.until and self.since > self.until:
            raise ValueError("since must not be after until")
        return self


PROMPT = """Split this question into a filter and a search string.

Question: {q}

Return JSON matching the schema. Leave a field null when the question does not bound
it: guessing a filter is worse than omitting one, because a wrong filter silently
returns nothing rather than failing. Today is 2026-08-25."""

for q in [
    "what did the tools tell us about billing since the start of August?",
    "has this happened before? checkout is timing out and the db seems to be blocking",
]:
    print(q[:64])
    print("   ->", extract(PROMPT.format(q=q), JournalFilter).model_dump())
    print()

what did the tools tell us about billing since the start of Augu


   -> {'query': 'what did the tools tell us about billing', 'source': 'tool_output', 'since': 20260801, 'until': None}

has this happened before? checkout is timing out and the db seem


   -> {'query': 'has this happened before? checkout is timing out and the db seems to be blocking', 'source': None, 'since': None, 'until': None}



The first question bounds two fields and both come back filled — `source` from "what did the *tools* tell us", `since` from "the start of August", a phrase no enum could enumerate turned into a number the store can compare. What is left, `billing`, is what gets ranked. The second bounds nothing, so every field stays `None`.

The validator is `03_tools`' — a rule about `since` and `until` together needs `model_validator`, and it raises for the same reason a bad tool argument does: `extract()` catches the `ValidationError`, hands the model the real message, and lets it try again. Same pattern, one layer up. What is genuinely new here is **what `None` means.**

In `03_tools`, a field the model leaves out is either a default it accepted or an error it has to fix; `required` and `extra="forbid"` exist to make that difference loud. Here every filter field is optional by design, and `None` is an instruction rather than a gap: *this question set no bound, so put no such clause in the `where` at all.* `since=None` does not mean "start date unknown", it means "no lower bound" — a complete answer, and often the right one.

**Which is why the prompt says to leave fields null when unsure, and that is not politeness.** A guessed filter is not a slightly-wrong filter. `{"about": "billing-api"}` on a question that was never about billing returns nothing at all, and an agent reads nothing as *this has never happened* rather than *my filter was wrong* — a wrong answer with no error attached to it. Omit the clause instead and you lose precision, which the ranking usually absorbs. That asymmetry is the whole reason the instruction is in the prompt: the two mistakes are not the same size.

### When a field's values cannot be listed

`source` has three values, so they go in the schema and the model picks one. `about` cannot work that way — here there are two services, but in a real system it is every repository or every customer, thousands of them, changing weekly. Too many for a schema.

But notice the filter does not need the model at all. A `where` takes a *list*, so the values worth filtering on can be found by lexical search and handed straight over.

`Journal.candidates()` runs **two** lexical searches and unions them, because each misses what the other catches.

The first searches the field's own vocabulary — the distinct strings `about` has ever held. That catches a question naming the value, and it is the only thing that can: `checkout-api` lives in a column and appears in nobody's prose, so no search over entry text will ever turn it up.

The second searches the entries and collects the values attached to the ones that rank well. That catches a question naming the *subject* without naming the value — "what happened with the schema cache?" matches no service name at all, but the entry it hits belongs to `billing-api`.

On this corpus the two barely overlap: the first answers questions naming "billing" or "checkout", the second answers questions about a "schema cache" or a "users table" — phrases that appear in no service name — and neither answers the other's. A lexical hit on a few entries is a seed — the values they carry become the filter, and the vector search then reaches everything else in those services, including entries the lexical search never found.

In [17]:
for q in ["anything about billing failing recently?",   # names the value
          "compare billing and checkout",               # names two values
          "what happened with the schema cache?",       # names neither -- found via entries
          "what happened with the users table?",        # ditto
          "has this happened before? timing out and blocking"]:
    print("%-50s -> %s" % (q[:48], journal.candidates("about", q)))

anything about billing failing recently?           -> ['billing-api']
compare billing and checkout                       -> ['checkout-api', 'billing-api']
what happened with the schema cache?               -> ['billing-api']
what happened with the users table?                -> ['checkout-api']
has this happened before? timing out and blockin   -> ['checkout-api']


A short list each time, and an empty one where the question named nothing.

A cutoff is needed because a value gets into those results on **any** shared word — `MATCH` decides membership before BM25 ranks anything — so the tail is always junk. The question is where to cut, and two obvious answers do not survive contact:

- **A fixed count.** Five is too few when a question names six services, and too many when it names one.
- **A score threshold.** BM25 scores depend on the corpus, so a number tuned against eight values is meaningless against eight thousand.

A *fraction of what matched* needs neither, which is what `keep` is.

None of this has to be precise, and that is the part worth holding on to. This list does not answer the question — it only widens or narrows what is allowed to come back, and resemblance decides the rest. A useless value costs one extra service's worth of rows in play. A missing value costs the answer outright: the right entry no longer satisfies the condition, so it cannot be returned however well it matches. Cut generously.

In [18]:
question = "anything about migrations going wrong on billing?"
services = journal.candidates("about", question)

print("filtering to:", services)
print()
for score, entry_id, text in journal.find_like(
        question, k=3, where={"about": {"$in": services}}):
    print(f"  {score:.4f}  {text[:58]}")

print()
print("unfiltered, for comparison:")
for score, entry_id, text in journal.find_like(question, k=3):
    print(f"  {score:.4f}  {text[:58]}")

filtering to: ['billing-api']



  0.8199  Billing runs its own migrations on a separate schedule.
  0.6280  PostgREST returned PGRST204: column 'invoice_ref' not foun
  0.5751  Priya should be paged when the error rate goes above two p

unfiltered, for comparison:
  0.8199  Billing runs its own migrations on a separate schedule.
  0.6863  The migration held a lock on the users table for about for
  0.6280  PostgREST returned PGRST204: column 'invoice_ref' not foun


Compare the two lists. Unfiltered, the second result is **checkout-api's** migration entry — a real migration problem, on the wrong service. The question said *on billing*, and without the condition that entry outranks two billing entries that do belong.

That is the whole value of the condition, and it is not subtle: resemblance alone has no way to know that "on billing" was a requirement rather than a hint. Chroma resolves it by prefiltering, so the checkout entry is never a candidate.

No model anywhere in that path. The lexical searches decided which values the condition should name, and resemblance ranked what qualified. The model is still doing the job only it can do, one step earlier — reading *"since the start of August"* and returning `20260801`, which no lexical lookup could. Listable fields go in the schema and the model picks from them; unlistable ones are found by search and never reach it.

## Question 5 — "who else touched this service, and what did they decide?"

This differs in kind from the four before it.

Every earlier answer was **inside a memory**. The timezone was in the profile; the failed deploy was one row; the error was one journal entry. An index found the right container and handed it over.

No entry contains the answer to this one. What exists is: Arun owns `checkout-api`; `checkout-api` calls `billing-api`; Meera owns `billing-api`. Nothing states that Meera is connected to `checkout-api`. That fact is **made of** three others, and the only way to get it is to walk from one to the next.

An edge is a different shape from a row, which is why this is a different store rather than another index.

In [19]:
org = Graph(reset=True)   # connects to Neo4j using the credentials in .env

(org.write("Priya", "owns", "checkout-api", since="2026-01-01", until="2026-07-01")
    .write("Arun", "owns", "checkout-api", since="2026-07-01")
    .write("checkout-api", "calls", "billing-api")
    .write("Meera", "owns", "billing-api")
    .write("Arun", "decided", "move migrations behind a feature flag", since="2026-08-13"))

print("edges:", len(org))

edges: 5


Now answer it — not with a picture of the graph, but with the two things asked: **who**, and **what did they decide**.

In [20]:
service = "checkout-api"

# someone who OWNS something -- the subject of an `owns` edge, not merely touched by one.
# Without the subject check, billing-api qualifies as a person, because "Meera owns
# billing-api" is an `owns` edge attached to it.
people = [n for n in org.connected(service, hops=2, at=TODAY)
          if any(e["relation"] == "owns" and e["subject"] == n
                 for e in org.about(n, at=TODAY))]

print(f"who else has touched {service}:")
for person in people:
    print(f"  {person:<6} - {Graph.phrase(org.path(service, person, at=TODAY))}")

print()
print("what they decided:")
for person in people:
    for edge in org.about(person, at=TODAY):
        if edge["relation"] == "decided":
            print(f"  {person}: {edge['object']} (from {edge['since']})")

who else has touched checkout-api:
  Arun   - Arun owns checkout-api
  Meera  - checkout-api calls billing-api -> Meera owns billing-api

what they decided:


  Arun: move migrations behind a feature flag (from 2026-08-13)


The second line is what makes it usable: **Meera is involved because `checkout-api` calls `billing-api`, which she owns.**

The path is the justification. "Meera, two hops" is true and impossible to check; the chain is something a person can verify and an agent can quote.

That took three queries, not one — `connected()` to find who is in range, `path()` for each chain, `about()` for the decisions — and none of them is a loop in Python. Here is the first, run directly rather than quoted, so you can see it produce the answer:

In [21]:
LIVE = ("($at IS NULL OR ((r.since IS NULL OR r.since <= $at) "
        "AND (r.until IS NULL OR r.until > $at)))")

CONNECTED = f"""
MATCH path = (n:Thing {{name: $start}})-[r:REL*1..2]-(other:Thing)
WHERE all(r IN relationships(path) WHERE {LIVE})
RETURN other.name AS name, min(length(path)) AS hops
ORDER BY hops, name
"""

for row in org.run(CONNECTED, start="checkout-api", at=TODAY):
    print(f"  {row['hops']} hop{'s' if row['hops'] > 1 else ' '}  {row['name']}")

  1 hop   Arun
  1 hop   billing-api
  2 hops  Meera
  2 hops  move migrations behind a feature flag


That is one round trip. `-[:REL*1..2]-` is the variable-length pattern — *follow between one and two edges from here* — and the database walks it next to the data. A hand-written version pulls one ring of neighbours, then the next, paying a round trip per ring.

Two details in that query are worth reading rather than skipping.

**`min(length(path))`** is there because a node can be reachable more than one way. Without it, a node two hops away that is *also* reachable by some longer route appears twice, at two different distances.

**`WHERE all(...)`** is easy to omit and expensive to get wrong. It requires **every** edge on the path to be live at the given date, so a chain cannot be assembled through a fact that had already stopped being true. Drop it and Priya — who handed the service over in July — starts appearing in answers about today, one hop away, with nothing marking her as historical.

## What "forgetting" means on a graph

Ownership of `checkout-api` changed hands in July. Priya had it before; Arun has it now.

The obvious move is to delete Priya's edge. That would be wrong, and it is the first appearance of a distinction notebook 3 is built around.

"Priya owns checkout-api" was never *false*. It was true, and then it stopped being true. Deleting the edge throws that difference away along with every question about the past. So edges carry `since` and `until`, and expire instead of vanishing:

In [22]:
for when in ["2026-03-01", TODAY]:
    print(f"attached to checkout-api on {when}:", org.neighbours("checkout-api", at=when))

print()
print("with no date given:", org.neighbours("checkout-api"))

attached to checkout-api on 2026-03-01: ['billing-api', 'Priya']
attached to checkout-api on 2026-08-22: ['billing-api', 'Arun']

with no date given: ['billing-api', 'Priya', 'Arun']


Both answers survive, and which you get depends on what you asked.

Now the last line. With no date supplied the walk returns **every edge that ever existed**, Priya included. That is deliberate and it is a trap: an agent asking "who owns this?" almost always means *now*, and would get a stale name alongside the current one. The alternative — defaulting to today — would make every saved output here depend on the day it ran.

There is no free answer. Deciding what a bare question means is a real design decision, and it is the same one that produces expiry, decay and contradiction in notebook 3.

## When one question needs two stores

Every question so far has had one right store. Plenty of real ones do not:

> *"Why did the latest billing deploy fail?"*

**Which** deploy is a filter-order-limit question, and only SQL answers it. **Why** it failed is prose, and only the journal has it. Neither half is optional and neither store can do the other's.

The shape that covers this is not "pick the right store". It is that each store takes a turn, and every turn does one of two jobs: it **answers**, or it **narrows what the next one may look at**.

In [23]:
from pipeline import FromGraph, FromJournal, FromRecords, Pipeline

reasons = Pipeline([
    FromRecords(records, kind="deploy", status="failed", limit=1),   # which one
    FromJournal(journal, k=3),                                       # why
])

for step, answer, narrowed in reasons.trace("why did it fail?"):
    print(step)
    if step == "records":
        for row in answer:
            print(f"    {row['at']}  {row['service']}  {row['text']}")
    else:
        for score, entry_id, text in answer:
            print(f"    {score:.4f}  #{entry_id}  {text[:58]}")
    if narrowed:
        print(f"    -> narrowed to {narrowed}")

records
    2026-08-19  billing-api  Rolled back. Error rate hit 4 percent.
    -> narrowed to {'about': ['billing-api'], 'since': '2026-08-05', 'until': '2026-09-02'}
journal
    0.5878  #5  PostgREST returned PGRST204: column 'invoice_ref' not foun
    0.5790  #6  Priya should be paged when the error rate goes above two p
    0.5706  #4  Billing runs its own migrations on a separate schedule.
    -> narrowed to {'about': ['billing-api']}


The SQL step found one row and handed on two things: the service, and a window of dates around it. The journal then ranked only inside that.

Worth running the same question without the first step, because the difference is the entire argument:

In [24]:
for score, entry_id, text in journal.find_like("why did it fail?", k=3):
    print(f"{score:.4f}  #{entry_id}  {journal.get(entry_id)['about']:<14} {text[:52]}")

0.6376  #1  checkout-api   The pool ran dry under peak traffic; raising max con
0.5943  #3  checkout-api   The migration held a lock on the users table for abo
0.5878  #5  billing-api    PostgREST returned PGRST204: column 'invoice_ref' no


Two of the three are about **checkout**, for a question asked about a billing failure. Resemblance has no way to know which failure "it" refers to, because the question does not say — that fact was in the records table, not in the sentence.

Notice also that the narrowing is a *window around* the event rather than a cutoff at it. That was a bug in the first version of this pipeline, and it is worth naming because it looks right: narrowing to `at >= 2026-08-19` excludes everything written **before** the deploy, which is where causes live. It returned a paging rule instead of an explanation.

### The narrowing is neutral; each store translates

What passes between steps is a small dict that belongs to none of the stores:

```python
{"about": ["billing-api"], "since": "2026-08-05", "until": "2026-09-02"}
```

Because the stores agree on nothing. Chroma wants integers and `$gte`; SQL wants strings and `>=`; Cypher wants a parameter. If the narrowing were written in any one of their dialects, every other step would be translating *out* of it:

In [25]:
from pipeline import _chroma_where

for narrowing in [{},
                  {"about": ["billing-api"]},
                  {"about": ["billing-api", "checkout-api"], "since": "2026-08-14"}]:
    print(f"{str(narrowing):62} -> {_chroma_where(narrowing)}")

{}                                                             -> None
{'about': ['billing-api']}                                     -> {'about': 'billing-api'}
{'about': ['billing-api', 'checkout-api'], 'since': '2026-08-14'} -> {'$and': [{'about': {'$in': ['billing-api', 'checkout-api']}}, {'at': {'$gte': 20260814}}]}


`None` means no restriction, a single value stays bare, and only two or more clauses become an `$and` — which is Chroma's requirement, not a style choice.

### What happens when a step narrows to nothing

A step that finds nothing has not necessarily failed. It may simply have nothing to contribute, and the question is whether the next step should then see **everything** or **nothing**.

Both are defensible and they are not interchangeable:

In [26]:
nothing_matches = FromRecords(records, service="does-not-exist", limit=1)
question = "has anything gone wrong with migrations?"

for label, journal_step in [("fail open  ", FromJournal(journal, k=2)),
                            ("fail closed", FromJournal(journal, k=2, fail_open=False))]:
    print(label)
    for step, answer, _ in Pipeline([nothing_matches, journal_step]).trace(question):
        if not answer:
            print(f"    {step}: nothing")
        elif isinstance(answer, str):
            print(f"    {step}: {answer}")
        else:
            for score, entry_id, text in answer:
                print(f"    {score:.4f}  #{entry_id}  {text[:52]}")
    print()

fail open  


    records: nothing
    0.7011  #3  The migration held a lock on the users table for abo
    0.6981  #4  Billing runs its own migrations on a separate schedu

fail closed
    records: nothing
    journal: stopped: nothing qualified and fail_open is False



**Fail open is the default here**, and it is the right one for a narrowing that was a *hint*. Nobody named a service, so searching every service is a reasonable reading of the question — and the two migration entries it finds are the answer.

**Fail closed is for a narrowing that was a *rule*.** A tenant filter, a permission filter, "only this user's memories": if the lookup comes back empty, widening the search returns somebody else's data. That is a bug, and no amount of relevance makes it not one.

The distinction is per step rather than global, because one pipeline can contain both — a mandatory tenant filter and an optional service hint.

### The other orders

The same two jobs, rearranged. These need the graph, so they are the compositions that only make sense once a store can be *walked*:

In [27]:
# a walk that narrows, then resemblance inside it
whose = Pipeline([
    FromGraph(org, start="Meera", hops=2),     # everything Meera is connected to
    FromJournal(journal, k=3),                 # what went wrong in there
])

# resemblance that narrows, then a walk out from it
who_else = Pipeline([
    FromJournal(journal, k=2),                 # the incidents that match
    FromGraph(org, hops=2),                    # who is attached to them
])

for title, pipe, q in [("Meera's services, then what went wrong", whose, "what went wrong?"),
                       ("the incidents, then who touched them", who_else,
                        "checkout is timing out and the database seems to be blocking")]:
    print(title)
    for step, answer, narrowed in pipe.trace(q):
        print(f"    {step:<9} -> {answer if step == 'graph' else [f'#{i}' for _, i, _ in answer]}")
        if narrowed:
            print(f"                 narrowed to {narrowed}")
    print()

Meera's services, then what went wrong


    graph     -> [('billing-api', 1), ('checkout-api', 2)]
                 narrowed to {'about': ['Meera', 'billing-api', 'checkout-api']}
    journal   -> ['#1', '#6', '#3']
                 narrowed to {'about': ['billing-api', 'checkout-api']}

the incidents, then who touched them
    journal   -> ['#3', '#1']
                 narrowed to {'about': ['checkout-api']}
    graph     -> [('Arun', 1), ('Priya', 1), ('billing-api', 1), ('Meera', 2), ('move migrations behind a feature flag', 2)]
                 narrowed to {'about': ['Arun', 'Meera', 'Priya', 'billing-api', 'checkout-api', 'move migrations behind a feature flag']}



### Who picks the pipeline

Every composition above is a literal list, written by hand. Nothing chose it at runtime, and that is the point rather than a shortcut.

There are three places this decision can live:

| | who composes | what it costs |
|---|---|---|
| one fixed pipeline | you | wrong for any question of a different shape |
| **several, model picks one** | **you write them, the model selects** | **a round trip before retrieval starts** |
| the model assembles the steps | the model | no cheap way to know the order it chose was sensible |

**The middle one.** An agent has a handful of recognisable question shapes, not an unbounded set, so a named pipeline per shape is something you can write once and test:

```python
diagnose  = Pipeline([FromRecords(...), FromJournal(...)])   # "why did X fail?"
ownership = Pipeline([FromGraph(...),   FromJournal(...)])   # "what is wrong with Meera's services?"
history   = Pipeline([FromJournal(...)])                     # "has this happened before?"
```

Each of those becomes a tool whose description says which question it answers, and `03_tools` already built that machinery. So the model chooses a **pipeline**, not a store — the composition stays a thing you wrote and can check, and the only judgement left to the model is recognising which shape of question it is looking at, which is the part it is actually good at.

The bottom row is the tempting one and it is worth being clear about why it is not here. A model asked to assemble steps will produce orders that are syntactically fine and retrieve nothing — narrow by a service nobody mentioned, walk from a node that does not exist — and the failure looks exactly like a question with no answer. A fixed pipeline that returns nothing tells you the store is missing something. A generated one tells you nothing at all.

Wiring these up as tools belongs with the loop, so `05_memory_in_the_loop.ipynb` does it.

Four stores, three orders, three different questions answered — and the only thing that changed between them is which store went first.

That is notebook 2's premise doing work rather than being asserted. Each store was chosen by **what would be asked of it**, so each one is good at exactly one kind of narrowing: a name, a filter, a walk, a resemblance. Composing them is choosing the order to narrow in.

**One thing no arrangement of these gets you.** Every store here answers by *selecting a few things* — one key, the matching rows, the nearest three, a neighbourhood. So *"what are the recurring themes across everything I remember?"* has no answer in any of them, because three entries cannot tell you about the other twenty-one.

Systems that answer that do it **before the question arrives**: cluster the memories, have a model summarise each cluster, and store the summaries. GraphRAG calls it a community report; the agent literature calls it **reflection**. It is a build-time cost rather than a query, and it is genuinely missing here rather than hiding in one of the four.

## None of this lives in the kernel

Every store writes somewhere that outlives the process: three to files under `store/`, and Neo4j to a server entirely off this machine.

In [28]:
print("profile:", KeyValue(STORE / "profile.json").read("timezone"))
print("records:", len(Records(STORE / "records.db")), "rows")

reopened = Journal(STORE / "journal")
print("journal:", len(reopened), "entries")

fresh = Graph()          # a new connection, not the one used above
print("edges  :", len(fresh))
fresh.close()

score, _id, text = reopened.find_like(question, k=1)[0]
print()
print(f"vector search after reopening: {score:.4f}  {text[:52]}")

org.close()   # a driver holds its connection pool open until told otherwise

profile: Asia/Kolkata
records: 6 rows
journal: 6 entries


edges  : 5

vector search after reopening: 0.7011  The migration held a lock on the users table for abo


That is the gap notebook 1 ended on, closed. The agent can be restarted and still answer all five questions.

`store/` stays in place on purpose, and the Neo4j instance keeps its edges — notebook 3 opens both.

## What you built

✓ Split two stores by **what each would be asked** — one filtered and counted, one searched — and kept the full text in both, so neither was made to look necessary by blinding the other. Authorship explains why they arrive in different shapes; it is not the criterion

✓ Answered "what is this user's timezone?" with a dict, and established that most long-term memory should stop there, since a store that returns *nearly* is a downgrade for a question with an exact answer

✓ Used SQL for filter, order, limit and count, and found the query with no equivalent anywhere else: no vector database has `COUNT`, at six rows or six million

✓ Counted the journal by `source` — told, observed, concluded — because the first thing you want when a memory turns out to be wrong is how the agent came by it

✓ Learned what FTS5 actually is: a boolean `MATCH` selecting candidates, then **BM25** ranking them. BM25 is the algorithm, FTS5 the engine implementing it

✓ Read the scores and found them **negative**, which is why `ORDER BY rank` ascending is best-first — and saw term frequency, saturation (3× the occurrences buys 1.19× the score) and length normalisation in three real numbers

✓ Measured why `$contains` is not a substitute: it matches `PGRST20`, a *different* code, as a prefix, and misses `pgrst204` on case

✓ Asked a question naming no field to filter on, and watched all five of its words return zero from full-text search — including `checkout`, which is in three entries' `about` field but in nobody's prose

✓ Confirmed filtering does not rescue it: `about="checkout-api"` returns three entries with no opinion about which explains a timeout

✓ Retrieved both prior contention incidents by meaning, then measured the cost: every score in a narrow band nowhere near zero, and `k=3` always returning three because a vector index cannot say *I don't have that*

✓ Asked for `PGRST205`, an error that never happened: FTS5 correctly returned nothing while the vector index confidently returned a different error, scored among the ones it gives correct answers

✓ Fused BM25 and cosine by reciprocal rank fusion — on **ranks**, because a negative BM25 score and a 0.54–0.67 cosine share no scale — and watched fusion make an answer **worse**, then traced it to the corpus rather than the method: `out` appears in one entry of six, so BM25 reads it as informative. At 60% document frequency the same term's weight collapses to ~1e-06, which is the rarity term doing the job a stopword list is standing in for here

✓ Found that Chroma implements this natively with `Rrf` over a BM25 sparse index, but `Search` raises `NotImplementedError: Search is not implemented for Local Chroma` — the feature exists in the product and not in the deployment, which is a very ordinary reason to run two engines

✓ Filtered on metadata before ranking, and hit the wrinkle that forces it: Chroma's ranges take numbers, so dates are stored twice

✓ Answered question 5 with a **path** — `checkout-api calls billing-api -> Meera owns billing-api` — because "Meera, two hops" is not something a person can check

✓ Expired an ownership edge rather than deleting it, because "Priya owns checkout-api" was true and then stopped being true, which is not the same as having been wrong

**Next:** `03_operations.ipynb` — write, update, read, retrieve and forget across all four, where only the signatures turn out to be portable.